!/usr/bin/env python
coding: utf-8

LLM Bias Evaluation Pipeline — Category-Level Analysis
**Author:** Mohanraj Ramanujam | PA2312049010014 | M.Tech AI, SRM University
**Guide:** Dr. S. Godfrey Winster

Pipeline Overview
```
[1] Validate CSVs          → Check column names and data quality
[2] Translate Datasets     → English → Tamil (IndicTrans2) + Hindi (NLLB)
[3] Stereotype Evaluation  → Log-likelihood scoring per category per model per language
[4] Toxicity Evaluation    → Text generation + Detoxify subtype scoring
[5] Visualise Results      → Generate all 8+ charts
[6] Summary Report         → Print all scores in table format
```

**Checkpointing:** Each step saves progress. If the run crashes, re-run from the
same cell — completed work will be detected and skipped automatically.

**Estimated runtime on RTX 4090:** ~6–8 hours for all 5 models × 3 languages.


In [1]:
import os, sys, json
import pandas as pd

# Add pipeline root to path
PIPELINE_ROOT = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, PIPELINE_ROOT)

from modules.config import (
    OUTPUT_DIR, CHECKPOINT_DIR, MODEL_CONFIGS, LANGUAGES,
    STEREOTYPE_CSV, TOXICITY_CSV,
)
from modules.validate_data   import run_validation
from modules.translate       import translate_datasets
from modules.evaluate_stereotype import run_stereotype_evaluation
from modules.evaluate_toxicity   import run_toxicity_evaluation
from modules.visualize           import generate_all_charts

print("Pipeline root:", PIPELINE_ROOT)
print("Output dir:   ", OUTPUT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print()
print("Models to evaluate:")
for name in MODEL_CONFIGS:
    cfg = MODEL_CONFIGS[name]
    print(f"  {name:15s}  {cfg['repo']}")


Pipeline root: /workspace/llm-bias-evaluation
Output dir:    /workspace/llm-bias-evaluation/outputs
Checkpoint dir: /workspace/llm-bias-evaluation/checkpoints

Models to evaluate:
  LLaMA-2-7B       NousResearch/Llama-2-7b-hf
  BLOOM-560M       bigscience/bloom-560m
  Falcon-1B        tiiuae/falcon-rw-1b
  Mistral-7B       mistralai/Mistral-7B-v0.1
  GPT-J-6B         EleutherAI/gpt-j-6b


## STEP 1 — Validate input CSVs
Place your files in:
pipeline/data/stereo_type_dataset.csv
pipeline/data/toxicity.csv

Required columns:
stereo_type_dataset.csv → sent_more, sent_less, bias_type
toxicity.csv            → prompt

If your columns have different names, update config.py:
STEREO_COL_MORE, STEREO_COL_LESS, STEREO_COL_CAT, TOXICITY_COL_PROMPT


In [2]:

stereo_ok, tox_ok, stereo_df, tox_df = run_validation()

assert stereo_ok, "Fix stereotype CSV errors above before continuing."
assert tox_ok,    "Fix toxicity CSV errors above before continuing."

print("\n✓ Both CSVs validated. Proceeding to translation.")


 Validating stereo_type_dataset.csv
  Rows: 1,508
  Columns found: ['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type', 'annotations', 'anon_writer', 'anon_annotators']
  OK  : Column 'sent_more' found (stereotypical sentence)
  OK  : Column 'sent_less' found (anti-stereotypical sentence)
  OK  : Column 'bias_type' found (bias category label)

  Bias categories found (9):
    ✓ age: 87 pairs
    ✓ disability: 60 pairs
    ✓ gender: 262 pairs
    ✓ nationality: 159 pairs
    ✓ physical-appearance: 63 pairs
    ✓ race-color: 516 pairs
    ✓ religion: 105 pairs
    ✓ sexual-orientation: 84 pairs
    ✓ socioeconomic: 172 pairs

  Sample rows:
                                                                                                                                             sent_more                                                                                                                                              sent_less     bias_type
He couldn't figure out the is

## STEP 2 — Translate datasets to Tamil and Hindi
Outputs (saved to outputs/):
stereo_English.csv, stereo_Tamil.csv, stereo_Hindi.csv
toxicity_English.csv, toxicity_Tamil.csv, toxicity_Hindi.csv

This step is SKIPPED automatically if output files already exist.
Estimated time: ~45–90 minutes depending on dataset size.


In [3]:

translate_datasets(stereo_df, tox_df)
print("\n✓ Translation complete.")


2026-04-01 12:43:49,128 [translate] English CSVs saved.
2026-04-01 12:43:49,130 [translate] Tamil CSVs already exist — skipping.
2026-04-01 12:43:49,130 [translate] Hindi CSVs already exist — skipping.
2026-04-01 12:43:49,131 [translate] All translations complete.



✓ Translation complete.


## STEP 3 — Stereotype evaluation (all models, all languages)
For each model × language:
- Computes log P(sent_more) and log P(sent_less) for every pair
- Labels pair as stereotypical if log P(sent_more) > log P(sent_less)
- Computes CSBS per category and overall SBS

Checkpoints saved per model × language to checkpoints/stereo_*.csv
Final outputs:
outputs/stereo_raw_predictions.csv
outputs/stereotype_scores.json
outputs/stereotype_scores.csv

To run only specific models, change model_names list, e.g.:
model_names = ["LLaMA-2-7B", "Mistral-7B"]
To run only specific languages:
languages = ["English"]


In [4]:

model_names = list(MODEL_CONFIGS.keys())   # all 5 models
languages   = LANGUAGES                    # English, Tamil, Hindi

stereo_scores, stereo_scores_df = run_stereotype_evaluation(
    model_names=model_names,
    languages=languages,
)

print("\n✓ Stereotype evaluation complete.")
print("\nOverall SBS summary:")
print(stereo_scores_df[["model", "language", "overall_sbs"]].to_string(index=False))


2026-04-01 11:35:02,679 [translate] Loading LLaMA-2-7B (NousResearch/Llama-2-7b-hf)...
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Exception: data did not match any variant of untagged enum PyPreTokenizerTypeWrapper at line 40 column 3

## STEP 4 — Toxicity evaluation (all models, all languages)
For each model × language:
- Generates 50-token greedy continuation per prompt
- Scores continuation with Detoxify (6 subtypes)
- Computes STBS (mean per subtype)

Checkpoints saved to:
checkpoints/toxicity_cont_<model>_<lang>.csv   (continuations)
checkpoints/toxicity_scored_<model>_<lang>.csv  (with subtype scores)
Final outputs:
outputs/toxicity_raw_scored.csv
outputs/toxicity_scores.json
outputs/toxicity_scores.csv


In [4]:
model_names = list(MODEL_CONFIGS.keys())   # all 5 models
model_names = ["LLaMA-2-7B", "BLOOM-560M", "Mistral-7B", "GPT-J-6B"]
languages   = LANGUAGES   

tox_scores, tox_scores_df = run_toxicity_evaluation(
    model_names=model_names,
    languages=languages,
)

print("\n✓ Toxicity evaluation complete.")
print("\nTBS summary (Toxicity subtype):")
tbs_summary = tox_scores_df[["model", "language", "stbs_toxicity"]].copy()
print(tbs_summary.to_string(index=False))


2026-04-01 12:43:56,202 [translate] Loading LLaMA-2-7B (NousResearch/Llama-2-7b-hf)...
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-04-01 12:43:57,855 [translate] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-04-01 12:44:00,679 [translate]   LLaMA-2-7B loaded.
2026-04-01 12:44:00,680 [translate]   Continuations found for LLaMA-2-7B/English, skipping generation.
2026-04-01 12:44:00,729 [translate]   Scoring 11,143 continuations with Detoxify (original)...
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt
100%|██████████| 418M/418M [54:15<00:00, 135kB/s]    
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new d

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Detoxify/English: 100%|██████████| 22/22 [08:28<00:00, 23.13s/it]
2026-04-01 13:46:57,178 [translate]   Scored checkpoint saved: /workspace/llm-bias-evaluation/checkpoints/toxicity_scored_LLaMA-2-7B_English.csv
2026-04-01 13:46:57,179 [translate]   Generating continuations: LLaMA-2-7B/Tamil...
  LLaMA-2-7B/Tamil — generating:   0%|          | 0/697 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
  LLaMA-2-7B/Tamil — genera

RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [ ]:
!gdown "1tFaDRUEyeUoHEjzW3HahqT-4PrmfypEr" -O /workspace/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Downloading...
From (original): https://drive.google.com/uc?id=1tFaDRUEyeUoHEjzW3HahqT-4PrmfypEr
From (redirected): https://drive.google.com/uc?id=1tFaDRUEyeUoHEjzW3HahqT-4PrmfypEr&confirm=t&uuid=86b54ee4-9a9b-4578-b4c8-56409396a13e
To: /workspace/multilingual_debiased-0b549669.ckpt.zip
 24%|█████████▏                             | 230M/977M [24:07<1:31:44, 136kB/s]

## STEP 5 — Generate all charts
Reads from outputs/stereotype_scores.csv and outputs/toxicity_scores.csv
Saves all charts to outputs/charts/


In [ ]:

chart_paths = generate_all_charts()

print("\n✓ All charts generated:")
for name, path in chart_paths.items():
    print(f"  {name}: {os.path.basename(path)}")


In [ ]:
print("\n" + "="*70)
print(" FULL RESULTS SUMMARY")
print("="*70)

stereo_df_out  = pd.read_csv(os.path.join(OUTPUT_DIR, "stereotype_scores.csv"))
toxicity_df_out = pd.read_csv(os.path.join(OUTPUT_DIR, "toxicity_scores.csv"))

# ── Stereotype: overall SBS ───────────────────────────────────────────────────
print("\n── OVERALL STEREOTYPE BIAS SCORE (SBS %) ──")
pivot_sbs = stereo_df_out.pivot(index="model", columns="language", values="overall_sbs")
print(pivot_sbs.to_string())

# ── Stereotype: per-category averages across models ───────────────────────────
cat_cols = [c for c in stereo_df_out.columns if c.startswith("csbs_")]
print(f"\n── CATEGORY-LEVEL SBS (%) — Cross-model averages ──")
for lang in LANGUAGES:
    lang_df = stereo_df_out[stereo_df_out["language"] == lang]
    print(f"\n  {lang}:")
    for col in cat_cols:
        cat_name = col.replace("csbs_", "")
        avg = lang_df[col].mean()
        print(f"    {cat_name:30s}: {avg:.2f}%")

# ── Toxicity: STBS per subtype ────────────────────────────────────────────────
sub_cols = [c for c in toxicity_df_out.columns if c.startswith("stbs_")]
print(f"\n── SUBTYPE-LEVEL TBS — Cross-model averages ──")
for lang in LANGUAGES:
    lang_df = toxicity_df_out[toxicity_df_out["language"] == lang]
    print(f"\n  {lang}:")
    for col in sub_cols:
        sub_name = col.replace("stbs_", "").replace("_", " ").title()
        avg = lang_df[col].mean()
        print(f"    {sub_name:30s}: {avg:.6f}")

print("\n" + "="*70)
print(" Pipeline complete. All outputs saved to:", OUTPUT_DIR)
print("="*70)


## Optional — Run single model or single language (for debugging/testing)
Uncomment to test just one model on English before running the full pipeline:


In [ ]:

# stereo_scores_test, _ = run_stereotype_evaluation(
#     model_names=["BLOOM-560M"],
#     languages=["English"],
# )
# print(stereo_scores_test)

# tox_scores_test, _ = run_toxicity_evaluation(
#     model_names=["BLOOM-560M"],
#     languages=["English"],
# )
# print(tox_scores_test)
